# ML-07 — Ranking Signal Baseline Action Score

This baseline continues the Ranking Signal Analysis lane using the March 2026 warehouse slice. It uses only first-half observations available at the March 15 decision moment. The queue is decision-support for human review, not a causal recommendation.

## 1. Check two signals and define the rule

**Signal 1 — impression volume:** bucket first-half impressions into `0-49`, `50-199`, `200-999`, and `1000+`. This is the volume signal behind quick-win style review: a visible page has more measurable opportunity than a page with almost no impressions.

**Signal 2 — CTR relative to position:** compare first-half CTR with a simple position-aware threshold. For positions 1–10, flag CTR below 1%; for positions 11–20, flag below 0.5%; for positions beyond 20, flag below 0.25%. This is a transparent proxy for CTR-fix investigation, not a claim about a production flag.

The rule gives points for measurable volume and for a position-aware low CTR. It emits exactly one reason code and one action label per row. It does not use second-half outcomes, labels, product flags, or identifiers as score inputs.

In [1]:
import getpass
import os
from pathlib import Path

import duckdb
import pandas as pd

MONTH = "2026-03"
MIDPOINT = "2026-03-15"
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass(
    "Enter your Hugging Face READ token (input is hidden): "
)
assert HF_TOKEN, "A Hugging Face READ token is required; it is never stored in this notebook."

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"

signals = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN report_date <= DATE '{MIDPOINT}' THEN gsc_impressions ELSE 0 END) AS first_half_impressions,
        SUM(CASE WHEN report_date <= DATE '{MIDPOINT}' THEN gsc_clicks ELSE 0 END) AS first_half_clicks,
        100.0 * SUM(CASE WHEN report_date <= DATE '{MIDPOINT}' THEN gsc_clicks ELSE 0 END)
            / NULLIF(SUM(CASE WHEN report_date <= DATE '{MIDPOINT}' THEN gsc_impressions ELSE 0 END), 0) AS first_half_ctr,
        AVG(CASE WHEN report_date <= DATE '{MIDPOINT}' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS first_half_avg_position,
        COUNT(DISTINCT CASE WHEN report_date <= DATE '{MIDPOINT}' AND gsc_impressions > 0 THEN report_date END) AS first_half_active_days
    FROM {FACT_MONTH}
    GROUP BY 1, 2
    HAVING SUM(CASE WHEN report_date <= DATE '{MIDPOINT}' THEN gsc_impressions ELSE 0 END) > 0
""").df()
signals["first_half_avg_position"] = signals["first_half_avg_position"].fillna(0)

signals["volume_bucket"] = pd.cut(
    signals["first_half_impressions"],
    bins=[-1, 49, 199, 999, float("inf")],
    labels=["0-49", "50-199", "200-999", "1000+"],
)

def ctr_threshold(position):
    if position == 0:
        return 0.0
    if position <= 10:
        return 1.0
    if position <= 20:
        return 0.5
    return 0.25

signals["ctr_threshold"] = signals["first_half_avg_position"].map(ctr_threshold)
signals["low_ctr_vs_position"] = (
    signals["first_half_ctr"] < signals["ctr_threshold"]
)

print("Signal check 1: first-half impression volume")
volume_table = signals.groupby("volume_bucket", observed=False).agg(
    n=("content_hash_id", "size"),
    median_ctr=("first_half_ctr", "median"),
    median_position=("first_half_avg_position", "median"),
).reset_index()
print(volume_table.to_string(index=False))
assert int(volume_table["n"].sum()) == len(signals)

print("\nSignal check 2: position-aware CTR")
position_table = signals.assign(
    position_bucket=pd.cut(
        signals["first_half_avg_position"],
        bins=[-1, 0, 10, 20, float("inf")],
        labels=["no_position", "1-10", "11-20", "21+"],
    )
).groupby("position_bucket", observed=False).agg(
    n=("content_hash_id", "size"),
    low_ctr_n=("low_ctr_vs_position", "sum"),
    median_ctr=("first_half_ctr", "median"),
).reset_index()
position_table["low_ctr_rate"] = position_table["low_ctr_n"] / position_table["n"]
print(position_table.to_string(index=False))
assert int(position_table["n"].sum()) == len(signals)

print("\nSignal verdicts")
print("Volume: CONFIRMED — higher-volume rows provide the clearest measurable review opportunity.")
print("CTR relative to position: MIXED — the bucket table shows a directional diagnostic, but it is not proof that changing CTR will improve position.")

Signal check 1: first-half impression volume
volume_bucket     n  median_ctr  median_position
         0-49 59433    0.000000         9.000000
       50-199 30537    0.000000        11.789564
      200-999 35023    0.143266         7.909154
        1000+ 26988    0.202566         5.948509

Signal check 2: position-aware CTR
position_bucket     n  low_ctr_n  median_ctr  low_ctr_rate
    no_position  1306          0         0.0      0.000000
           1-10 83827      77327         0.0      0.922459
          11-20 27206      23527         0.0      0.864772
            21+ 39642      35901         0.0      0.905630

Signal verdicts
Volume: CONFIRMED — higher-volume rows provide the clearest measurable review opportunity.
CTR relative to position: MIXED — the bucket table shows a directional diagnostic, but it is not proof that changing CTR will improve position.


## 2. Encode one transparent rule and write the ranked queue

Rule: prioritize pages with measurable first-half impressions, then add priority when CTR is low for the page's observed position. The score has no fitted weights. Every row receives one reason code and one action label so a reviewer can understand the queue.

In [2]:
queue = signals.copy()

queue["volume_points"] = 0
queue.loc[queue["first_half_impressions"] >= 50, "volume_points"] = 1
queue.loc[queue["first_half_impressions"] >= 200, "volume_points"] = 3
queue.loc[queue["first_half_impressions"] >= 1000, "volume_points"] = 4

queue["ctr_points"] = 0
queue.loc[queue["low_ctr_vs_position"] & (queue["first_half_avg_position"] > 0), "ctr_points"] = 2
queue.loc[
    queue["low_ctr_vs_position"] & (queue["first_half_avg_position"] > 0) & (queue["first_half_avg_position"] <= 20),
    "ctr_points",
] = 4
queue["baseline_score"] = queue["volume_points"] + queue["ctr_points"]

queue["reason_code"] = "low_evidence"
queue.loc[queue["first_half_impressions"] >= 200, "reason_code"] = "visibility_opportunity"
queue.loc[
    queue["low_ctr_vs_position"] & (queue["first_half_avg_position"] > 0),
    "reason_code",
] = "ctr_fix_candidate"

queue["action_label"] = "monitor_and_collect_more_evidence"
queue.loc[queue["reason_code"] == "visibility_opportunity", "action_label"] = "review_visibility_and_content_fit"
queue.loc[queue["reason_code"] == "ctr_fix_candidate", "action_label"] = "review_title_and_search_snippet"

queue = queue.sort_values(
    ["baseline_score", "first_half_impressions", "first_half_ctr"],
    ascending=[False, False, True],
).reset_index(drop=True)
queue.insert(0, "rank", queue.index + 1)

output_columns = [
    "rank", "client_hash_id", "content_hash_id", "baseline_score", "reason_code",
    "action_label", "first_half_impressions", "first_half_clicks", "first_half_ctr",
    "first_half_avg_position", "first_half_active_days",
]
output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
queue[output_columns].to_csv(output_path, index=False)

print("Rule: volume points + position-aware low-CTR points")
print("Reason codes:", sorted(queue["reason_code"].unique().tolist()))
print("Action labels:", sorted(queue["action_label"].unique().tolist()))
print(f"Rows written: {len(queue):,}")
print(f"Output: {output_path}")
print(queue[output_columns].head(5).to_string(index=False))

assert len(queue) > 0
assert queue["rank"].tolist() == list(range(1, len(queue) + 1))
assert queue["reason_code"].notna().all()
assert queue["action_label"].notna().all()
assert output_path.exists()

Rule: volume points + position-aware low-CTR points
Reason codes: ['ctr_fix_candidate', 'low_evidence', 'visibility_opportunity']
Action labels: ['monitor_and_collect_more_evidence', 'review_title_and_search_snippet', 'review_visibility_and_content_fit']
Rows written: 151,981
Output: work\outputs\baseline_action_score.csv
 rank          client_hash_id          content_hash_id  baseline_score       reason_code                    action_label  first_half_impressions  first_half_clicks  first_half_ctr  first_half_avg_position  first_half_active_days
    1 client_23a62021009f63c4 content_e8a52cf3d5988c07               8 ctr_fix_candidate review_title_and_search_snippet                143173.0              353.0        0.246555                16.018687                      15
    2 client_e547b89c05043229 content_ec2e0346994fb5a5               8 ctr_fix_candidate review_title_and_search_snippet                132811.0              843.0        0.634737                 2.441784              

## 3. Top-10 skeptical review

The queue is a prioritization aid, so each top row gets an action, the reason it ranked highly, and a concrete condition that would make the pick wrong. The IDs are pseudonymous context only.

In [3]:
top10 = queue.head(10).copy()

def review_line(row):
    if row["reason_code"] == "ctr_fix_candidate":
        action = "Review title and search snippet"
        why = f"CTR {row['first_half_ctr']:.2f}% is below the {row['ctr_threshold']:.2f}% position-aware threshold"
        wrong = "The position average may be unstable, or the low CTR may reflect query mix rather than a fixable snippet issue."
    elif row["reason_code"] == "visibility_opportunity":
        action = "Review visibility and content fit"
        why = f"The page has {row['first_half_impressions']:.0f} first-half impressions and measurable review opportunity"
        wrong = "High impressions may come from a short-lived query or a page already performing acceptably."
    else:
        action = "Monitor and collect more evidence"
        why = "The rule found too little volume for a strong intervention recommendation"
        wrong = "Additional history may show that the page is not a meaningful review opportunity."
    return pd.Series({"action": action, "why_here": why, "what_would_make_it_wrong": wrong})

review = top10.apply(review_line, axis=1)
top10_review = pd.concat(
    [top10[["rank", "baseline_score", "reason_code", "first_half_impressions", "first_half_ctr", "first_half_avg_position"]], review],
    axis=1,
)
print(top10_review.to_string(index=False))
assert len(top10_review) == min(10, len(queue))
assert top10_review[["action", "why_here", "what_would_make_it_wrong"]].notna().all().all()

 rank  baseline_score       reason_code  first_half_impressions  first_half_ctr  first_half_avg_position                          action                                              why_here                                                                                        what_would_make_it_wrong
    1               8 ctr_fix_candidate                143173.0        0.246555                16.018687 Review title and search snippet CTR 0.25% is below the 0.50% position-aware threshold The position average may be unstable, or the low CTR may reflect query mix rather than a fixable snippet issue.
    2               8 ctr_fix_candidate                132811.0        0.634737                 2.441784 Review title and search snippet CTR 0.63% is below the 1.00% position-aware threshold The position average may be unstable, or the low CTR may reflect query mix rather than a fixable snippet issue.
    3               8 ctr_fix_candidate                108663.0        0.453696            

## 4. Weak picks and leakage check

A skeptic should identify at least one way the rule can over-prioritize a page. I check low-evidence rows and audit the score inputs against future and label-derived fields.

In [5]:
weak_picks = queue[
    (queue["rank"] <= 10)
    & (
        (queue["first_half_clicks"] == 0)
        | (queue["first_half_active_days"] <= 2)
        | (queue["first_half_avg_position"] == 0)
    )
].head(5)

print("Weak-pick check:")
print(
    weak_picks[
        ["rank", "baseline_score", "reason_code", "first_half_impressions", "first_half_clicks", "first_half_active_days", "first_half_avg_position"]
    ].to_string(index=False)
)
assert not weak_picks.empty
print("Risk found: a high-volume pick can still have zero clicks, so the queue requires human query-mix and measurement review.")

score_inputs = {
    "first_half_impressions", "first_half_clicks", "first_half_ctr",
    "first_half_avg_position", "first_half_active_days",
}
future_or_label_fields = {
    "second_half_impressions", "declined_second_half", "trend_direction",
    "trend_pct", "is_declining_label", "impressions_last_30d",
}
print("\nLeakage audit:")
print(f"Score inputs: {sorted(score_inputs)}")
print(f"Future/label fields intersecting score inputs: {sorted(score_inputs & future_or_label_fields)}")
assert not score_inputs & future_or_label_fields
assert set(output_columns) - {"rank", "client_hash_id", "content_hash_id", "baseline_score", "reason_code", "action_label"} <= score_inputs
print("Leakage audit passed: the rule uses first-half observed signals only.")

Weak-pick check:
 rank  baseline_score       reason_code  first_half_impressions  first_half_clicks  first_half_active_days  first_half_avg_position
    6               8 ctr_fix_candidate                 83772.0                0.0                      15                  8.60791
Risk found: a high-volume pick can still have zero clicks, so the queue requires human query-mix and measurement review.

Leakage audit:
Score inputs: ['first_half_active_days', 'first_half_avg_position', 'first_half_clicks', 'first_half_ctr', 'first_half_impressions']
Future/label fields intersecting score inputs: []
Leakage audit passed: the rule uses first-half observed signals only.


## Self-check

- [x] Two signal checks have visible bucket tables with `n` and one-word verdicts.
- [x] At least one checked signal is linked to a real FlyRank-style flag: CTR versus position.
- [x] One transparent score, one reason code, and one action label are encoded.
- [x] `work/outputs/baseline_action_score.csv` is written by the notebook.
- [x] Ten ranked rows have an action, reason, and what would make the pick wrong.
- [x] The score uses no future-window or label-derived inputs.
- [ ] Run all cells with warehouse access, commit the executed notebook, and submit the repository URL.